# Tái hiện pipeline WHU-LX XGB-DQN

Notebook này là bản tự chứa của pipeline WHU-LX XGB-DQN. Toàn bộ hàm xử lý dữ liệu, huấn luyện XGBoost, xây dựng DQN, rollout, đánh giá và lưu kết quả đều nằm trực tiếp trong notebook, nên không còn phụ thuộc vào `scripts/reproduce_whulx_pipeline.py`.

Mục tiêu của notebook là chạy pipeline theo phạm vi toàn bộ dataset:

- Đọc dữ liệu `data/Cleaned_data.csv` đã được đưa vào repo.
- Làm sạch và mã hóa dữ liệu để mô hình học máy xử lý được.
- Huấn luyện XGBoost làm mô hình chuyển trạng thái nhiệt độ trong nhà.
- Huấn luyện DQN bằng cách đi qua nhiều ngày hoàn chỉnh trong dataset, thay vì chỉ dùng một lát cắt dữ liệu nhỏ.
- Đánh giá chính sách DQN trên toàn bộ các ngày hoàn chỉnh và so sánh với human baseline cùng các baseline cố định.
- Lưu kết quả ra `artifacts/outputs/whulx_reproduction/` để có thể đưa lên Git hoặc dùng lại trong báo cáo.

Điểm khác so với bản notebook trước: notebook này tập trung vào pipeline toàn bộ, từ train đến evaluate trên toàn bộ tập ngày có thể đánh giá.


## 1. Chuẩn bị thư viện và đường dẫn

Pipeline cần các nhóm thư viện chính:

- `pandas`, `numpy`: đọc và xử lý dữ liệu dạng bảng.
- `scikit-learn`: chia train/test, mã hóa nhãn và tính metric.
- `xgboost`: huấn luyện mô hình dự đoán biến thiên nhiệt độ trong nhà.
- `tensorflow`: xây dựng và huấn luyện mạng DQN.

Dữ liệu được đọc trực tiếp từ `data/Cleaned_data.csv`. Đây là bản sao dữ liệu từ nguồn WHU-LX đã được đưa vào repo để notebook tự chạy được sau khi clone.

In [ ]:
import json
import math
import random
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "artifacts" / "outputs" / "whulx_reproduction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "Cleaned_data.csv"

np.random.seed(2022)
random.seed(2022)
tf.random.set_seed(2022)

print("Project root:", PROJECT_ROOT)
print("Dữ liệu:", DATA_PATH)
print("Thư mục lưu kết quả:", OUT_DIR)

## 2. Định nghĩa vùng tiện nghi nhiệt ASHRAE

Reward của DQN cần biết nhiệt độ trong nhà có đang nằm trong vùng tiện nghi hay không. Repo WHU-LX dùng vùng tiện nghi thích nghi theo ASHRAE.

Ý tưởng chính:

- Khi nhiệt độ ngoài trời thấp hơn hoặc bằng ngưỡng dưới, dùng dải tiện nghi thấp.
- Khi nhiệt độ ngoài trời cao hơn hoặc bằng ngưỡng trên, dùng dải tiện nghi cao.
- Nếu nhiệt độ ngoài trời nằm giữa hai ngưỡng, nội suy tuyến tính để lấy dải tiện nghi.

Trong notebook này, hàm `comfort_bounds()` trả về bốn mốc của dải tiện nghi; hàm `comfort_ok()` kiểm tra nhiệt độ trong nhà có nằm trong khoảng tiện nghi chính hay không.

In [ ]:
STANDARD_BANDS = {
    "ASHRAE": [[17.4, 18.4, 23.4, 24.4], [23.6, 24.6, 29.6, 30.6], [10, 30]]
}


def comfort_bounds(outdoor_temp, standard="ASHRAE"):
    l11, l12, u12, u11 = STANDARD_BANDS[standard][0]
    l21, l22, u22, u21 = STANDARD_BANDS[standard][1]
    t1, t2 = STANDARD_BANDS[standard][2]

    if outdoor_temp <= t1:
        return l11, l12, u12, u11
    if outdoor_temp >= t2:
        return l21, l22, u22, u21

    increase_l = (outdoor_temp - t1) * (l21 - l11) / (t2 - t1)
    increase_u = (outdoor_temp - t1) * (u21 - u11) / (t2 - t1)
    return l11 + increase_l, l12 + increase_l, u12 + increase_u, u11 + increase_u


def comfort_ok(indoor_temp, outdoor_temp):
    _, lower, upper, _ = comfort_bounds(float(outdoor_temp))
    return lower <= float(indoor_temp) <= upper

## 3. Thiết kế reward và action space

DQN cần hai thành phần quan trọng: action và reward.

### Action space

Pipeline gốc dùng 24 action:

- `0`: tắt điều hòa, đóng cửa sổ.
- `1-11`: bật điều hòa, đóng cửa sổ, đặt nhiệt độ mục tiêu từ 20 đến 30 độ C.
- `12`: tắt điều hòa, mở cửa sổ.
- `13-23`: bật điều hòa và mở cửa sổ, đặt nhiệt độ mục tiêu từ 20 đến 30 độ C.

### Reward

Reward gồm hai phần:

- Phạt sai lệch tiện nghi: nếu nhiệt độ trong nhà nằm ngoài vùng tiện nghi thì phạt theo bình phương khoảng cách.
- Phạt năng lượng: action có bật điều hòa sẽ bị trừ chi phí; action vừa bật điều hòa vừa mở cửa sổ bị phạt nặng hơn.

Vì chi phí điều hòa trong reward khá lớn, agent có xu hướng ưu tiên các action không bật điều hòa nếu vẫn giữ được comfort tương đối ổn.

In [ ]:
def calculate_reward(state, action, next_state):
    indoor_temp = float(next_state[0])
    outdoor_temp = float(next_state[2])
    _, lower, upper, _ = comfort_bounds(outdoor_temp)

    if lower <= indoor_temp <= upper:
        reward = 0.0
    elif indoor_temp < lower:
        reward = -((indoor_temp - lower) ** 2)
    else:
        reward = -((indoor_temp - upper) ** 2)

    if action in {0, 12}:
        reward += 0.0
    elif action > 12:
        reward -= 2 * (60 * 0.87 * 1)
    else:
        reward -= 60 * 0.87 * 1
    return float(reward)


def map_action_to_dataframe(action):
    action = int(action)
    target_temp, ac_status, window_status, c_last_time, w_last_time = 0, 0, 0, 0, 0

    if action == 0:
        pass
    elif 0 < action < 12:
        target_temp = 19 + action
        ac_status = 1
        c_last_time = 60
    elif action == 12:
        window_status = 1
        w_last_time = 60
    else:
        target_temp = action + 7
        ac_status = 1
        window_status = 1
        c_last_time = 60
        w_last_time = 60

    return target_temp, ac_status, window_status, c_last_time, w_last_time

## 4. Đọc và tiền xử lý dữ liệu

Dữ liệu đầu vào là `Cleaned_data.csv` từ WHU-LX. Dù tên file là cleaned, pipeline vẫn làm thêm một số bước để đảm bảo mô hình học được:

- Đọc CSV với encoding `gbk` vì dữ liệu gốc dùng encoding này.
- Chuyển `Date_Time` sang kiểu thời gian.
- Loại bỏ missing value.
- Loại bỏ giá trị lỗi `-999`.
- Mã hóa các cột dạng chuỗi sang số bằng `LabelEncoder`.

Kết quả gồm `raw` là dữ liệu gốc và `data` là dữ liệu đã xử lý để huấn luyện.

In [ ]:
def load_and_prepare_data(data_path):
    raw = pd.read_csv(data_path, encoding="gbk")
    data = raw.copy()

    data["Date_Time"] = pd.to_datetime(data["Date_Time"])
    data = data.dropna()
    data = data[data != -999].dropna()

    for col in data.columns:
        if data[col].dtype == "object":
            encoder = LabelEncoder()
            data[col] = encoder.fit_transform(data[col])

    return raw, data


raw, data = load_and_prepare_data(DATA_PATH)

print("Kích thước dữ liệu gốc:", raw.shape)
print("Kích thước sau xử lý:", data.shape)
data.head()

## 5. Huấn luyện XGBoost để dự đoán chuyển trạng thái

Trong pipeline này, XGBoost không phải là mô hình điều khiển. Nó đóng vai trò như một **mô hình môi trường gần đúng**.

Cụ thể:

- Input của XGBoost là trạng thái hiện tại cộng với thông tin action/trạng thái điều khiển.
- Target là `Differ_Indoor_Temp`, tức độ thay đổi nhiệt độ trong nhà ở bước tiếp theo.
- Khi DQN chọn action, XGBoost dự đoán `Differ_Indoor_Temp` để cập nhật `Indoor_Temp` tiếp theo.

Các metric theo dõi:

- MAE: sai số tuyệt đối trung bình.
- RMSE: sai số bình phương trung bình lấy căn, nhạy hơn với lỗi lớn.
- R2: mức độ mô hình giải thích được biến thiên của target.

In [ ]:
def train_xgboost(data, device="cpu"):
    x_data = data.drop(
        ["Next_Indoor_Temp", "Next_Indoor_RH", "Date_Time", "Study_ID", "Differ_Indoor_Temp", "ID"],
        axis=1,
    )
    y_data = data["Differ_Indoor_Temp"]

    x_train, x_test, y_train, y_test = train_test_split(
        x_data, y_data, test_size=0.2, random_state=2022
    )

    model = xgb.XGBRegressor(
        random_state=2000,
        verbosity=0,
        n_jobs=-1,
        tree_method="hist",
        device=device,
        max_depth=5,
        learning_rate=0.23474,
        n_estimators=500,
    )
    model.fit(x_train, y_train)

    pred = model.predict(x_test)
    mse = mean_squared_error(y_test, pred)
    metrics = {
        "x_shape": list(x_data.shape),
        "train_shape": list(x_train.shape),
        "test_shape": list(x_test.shape),
        "mae": float(mean_absolute_error(y_test, pred)),
        "rmse": float(math.sqrt(mse)),
        "r2": float(r2_score(y_test, pred)),
    }
    return model, metrics


model_xgb, xgb_metrics = train_xgboost(data, device="cpu")
xgb_metrics

## 6. Tách dữ liệu thành các cửa sổ ngày

Dataset WHU-LX là chuỗi quan sát theo thời gian. Để DQN tương tác với dữ liệu, ta cần cắt chuỗi này thành các cửa sổ theo ngày. Mỗi cửa sổ ngày được xem như một episode nhỏ của môi trường điều khiển.

### Các định nghĩa cần biết

- **Cửa sổ ngày**: một đoạn dữ liệu liên tiếp đại diện cho một ngày quan sát. Trong notebook, mỗi cửa sổ được lấy bằng `start_index : start_index + 23`.
- **Timestep**: một bước thời gian trong cửa sổ ngày. Mỗi timestep có nhiệt độ trong nhà, nhiệt độ ngoài trời, độ ẩm, thời tiết và trạng thái điều khiển.
- **Khung điều khiển**: phần thời gian agent thực sự chọn action. Pipeline dùng khung từ step 6 đến step 17, tương ứng 12 quyết định điều khiển trong một ngày.
- **State của DQN**: vector trạng thái mà agent nhìn thấy trước khi chọn action. Ở đây state chính gồm 8 biến: `Indoor_Temp`, `Indoor_RH`, `Outdoor_Temp`, `Outdoor_RH`, `Rain`, `Cloud`, `Windspeed`, `Hour`.
- **Feature của XGBoost**: bộ biến đầy đủ hơn dùng để dự đoán `Differ_Indoor_Temp`. Ngoài các biến môi trường, bảng này còn có `Target_Temp`, `AC_Status`, `Window_Status`, `CLast_Time`, `WLast_Time` và các biến thời gian tích lũy.
- **Transition**: quá trình chuyển từ `state` sang `next_state` sau khi agent chọn một action. Trong notebook, transition không lấy trực tiếp từ môi trường thật mà được mô phỏng gần đúng bằng XGBoost.

Hàm `choose_day()` nhận `start_index` và trả về hai bảng:

- `data_test`: bảng trạng thái mà DQN quan sát.
- `xgboost_test`: bảng feature đầy đủ hơn để XGBoost dự đoán `Differ_Indoor_Temp`.

Mặc dù tên hàm là `choose_day()`, notebook dùng nó như tiện ích cắt dữ liệu theo ngày trong train và evaluate, không dùng để viết một phần phân tích riêng cho một ngày cụ thể. Hàm này chỉ đóng vai trò để:

- lấy từng ngày trong quá trình huấn luyện DQN,
- lấy từng ngày trong quá trình đánh giá toàn bộ dataset,
- đảm bảo logic rollout dùng cùng định dạng dữ liệu ở mọi ngày.

Biến `max_complete_days` cho biết dataset hiện có bao nhiêu ngày hoàn chỉnh có thể rollout. Đây là phạm vi train/evaluate chính của notebook.


In [ ]:
def choose_day(start_index, data):
    data_test_00 = data.iloc[start_index : start_index + 23].copy()
    data_test_0a = data_test_00.reset_index(drop=True)
    data_test_0 = data_test_00.reset_index(drop=True)

    data_test_0.at[0, "CLast_Time_T"] = (
        data_test_0a.iloc[0]["CLast_Time_T"] - data_test_0a.iloc[0]["AC_Status"] * 60
    )
    data_test_0.at[0, "WLast_Time_T"] = (
        data_test_0a.iloc[0]["WLast_Time_T"] - data_test_0a.iloc[0]["Window_Status"] * 60
    )

    data_test = data_test_0[
        [
            "Indoor_Temp",
            "Indoor_RH",
            "Outdoor_Temp",
            "Outdoor_RH",
            "Rain",
            "Cloud",
            "Windspeed",
            "Hour",
            "Next_Outdoor_Temp",
            "Next_Outdoor_RH",
        ]
    ].copy()

    xgboost_test = data_test_0.drop(
        ["Next_Indoor_Temp", "Next_Indoor_RH", "Date_Time", "Study_ID", "Differ_Indoor_Temp", "ID"],
        axis=1,
    )
    return data_test, xgboost_test


max_complete_days = max(1, (len(data) - 24) // 24)
print(f"Số ngày hoàn chỉnh có thể rollout: {max_complete_days}")


## 7. Xây dựng Replay Buffer và mạng DQN

DQN là một mạng neural network nhận state và trả về Q-value cho từng action. Nói cách khác, mạng DQN học cách ước lượng: nếu đang ở trạng thái hiện tại thì mỗi action có giá trị dài hạn tốt hay xấu như thế nào.

### Các định nghĩa cần biết

- **Agent**: bộ điều khiển học tăng cường. Trong notebook này, agent là DQN.
- **Environment**: môi trường mà agent tương tác. Ở đây environment không phải EnergyPlus trực tiếp, mà là dữ liệu ngày kết hợp với mô hình XGBoost để dự đoán trạng thái kế tiếp.
- **Action**: quyết định điều khiển HVAC/cửa sổ. Pipeline dùng 24 action, gồm tắt AC, mở cửa, bật AC ở các setpoint khác nhau hoặc kết hợp bật AC và mở cửa.
- **Reward**: điểm thưởng/phạt sau mỗi action. Reward phản ánh hai mục tiêu: giữ nhiệt độ trong vùng tiện nghi và hạn chế dùng điều hòa.
- **Q-value**: giá trị kỳ vọng dài hạn của một action tại một state. DQN chọn action có Q-value cao nhất khi không còn khám phá ngẫu nhiên.
- **Policy**: chính sách chọn action. Trong giai đoạn train, policy là epsilon-greedy; trong giai đoạn evaluate, policy chọn action có Q-value cao nhất.
- **Replay buffer**: bộ nhớ lưu các transition `(state, action, next_state, reward)` để huấn luyện lại theo batch ngẫu nhiên.
- **Batch size**: số transition lấy ra từ replay buffer trong một lần cập nhật mạng.
- **Gamma**: hệ số chiết khấu reward tương lai. Gamma càng cao thì agent càng quan tâm đến kết quả dài hạn.
- **Epsilon**: xác suất chọn action ngẫu nhiên. Epsilon cao giúp agent khám phá nhiều action; epsilon thấp giúp agent khai thác chính sách đã học.

Kiến trúc mạng DQN trong notebook:

- Dense 64, ReLU.
- Dense 64, ReLU.
- Dense 24, linear output, mỗi node tương ứng một action.

Replay buffer giúp giảm tương quan giữa các mẫu liên tiếp trong rollout. Nếu huấn luyện trực tiếp theo đúng thứ tự thời gian, các mẫu gần nhau thường quá giống nhau và mạng dễ học lệch. Lấy mẫu ngẫu nhiên từ replay buffer làm quá trình học ổn định hơn.


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, next_state, reward):
        self.buffer.append((state, action, next_state, reward))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)


class DQN(tf.keras.Model):
    def __init__(self, num_actions):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(64, activation="relu")
        self.dense2 = tf.keras.layers.Dense(64, activation="relu")
        self.output_layer = tf.keras.layers.Dense(num_actions, activation="linear")

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        return self.output_layer(x)

## 8. Hàm cập nhật Q-network và chính sách epsilon-greedy

Ở mỗi lần cập nhật, target Q được tính theo công thức DQN cơ bản:

`target = reward + gamma * max_a Q(next_state, a)`

Sau đó mô hình chỉ so sánh target với Q-value của action đã thực sự chọn. Hàm `epsilon_greedy_policy()` dùng để cân bằng giữa khám phá và khai thác:

- Với xác suất `epsilon`, chọn action ngẫu nhiên.
- Ngược lại, chọn action có Q-value cao nhất.

In [ ]:
def update_q_network(q_network, replay_buffer, optimizer, loss_fn, gamma, num_actions, batch_size):
    states, actions, next_states, rewards = zip(*replay_buffer.sample(batch_size))
    states = tf.convert_to_tensor(np.array(states), dtype=tf.float32)
    next_states = tf.convert_to_tensor(np.array(next_states), dtype=tf.float32)
    actions = tf.convert_to_tensor(np.array(actions), dtype=tf.int32)
    rewards = tf.convert_to_tensor(np.array(rewards), dtype=tf.float32)

    with tf.GradientTape() as tape:
        q_values = q_network(states)
        target_q_values = q_network(next_states)
        target_q_values = rewards + gamma * tf.reduce_max(target_q_values, axis=1)
        mask = tf.one_hot(actions, num_actions)
        q_action = tf.reduce_sum(q_values * mask, axis=1)
        loss = loss_fn(target_q_values, q_action)

    grads = tape.gradient(loss, q_network.trainable_variables)
    optimizer.apply_gradients(zip(grads, q_network.trainable_variables))
    return float(loss.numpy())


def epsilon_greedy_policy(q_network, state, epsilon, num_actions):
    if np.random.rand() < epsilon:
        return int(np.random.randint(num_actions))
    q_values = q_network(np.array([state], dtype=np.float32))
    return int(np.argmax(q_values[0]))

## 9. Rollout một cửa sổ ngày bằng XGBoost + DQN

`rollout_day()` là lõi mô phỏng của pipeline. Mỗi lần gọi hàm tương ứng với một cửa sổ ngày trong dataset.

Quy trình bên trong gồm hai phần:

1. **Warm-up trước khung điều khiển**: các bước đầu ngày được cập nhật bằng XGBoost để tạo trạng thái hợp lý trước khi agent bắt đầu ra quyết định.
2. **Khung điều khiển 6-17h**: tại mỗi bước, DQN chọn action, action được ghi vào feature điều khiển, XGBoost dự đoán thay đổi nhiệt độ trong nhà, sau đó notebook tính reward và cập nhật replay buffer nếu đang huấn luyện.

Hàm này cũng hỗ trợ `fixed_action`. Nhờ đó, cùng một logic rollout có thể dùng cho nhiều baseline:

- luôn tắt AC,
- luôn mở cửa,
- luôn bật AC ở một setpoint cố định,
- hoặc dùng chính sách DQN đã học.

Điểm quan trọng: XGBoost không tối ưu quyết định điều khiển. Nó chỉ đóng vai trò môi trường chuyển trạng thái gần đúng. DQN mới là phần học chính sách chọn action.


In [ ]:
def rollout_day(
    model_xgb,
    q_network,
    data_test,
    xgboost_test,
    epsilon,
    train=False,
    replay_buffer=None,
    train_cfg=None,
    fixed_action=None,
):
    data_pre_test = data_test.copy()
    xgboost_pre_test = xgboost_test.copy()
    num_features = 8
    actions = []
    rewards = []
    losses = []

    for step in range(6):
        xgboost_pre_test.loc[
            step,
            ["Target_Temp", "AC_Status", "Window_Status", "CLast_Time", "WLast_Time", "CLast_Time_T", "WLast_Time_T"],
        ] = 0
        hour_row_df = pd.DataFrame(xgboost_pre_test.iloc[step]).T
        next_differ_temp = model_xgb.predict(hour_row_df)[0]
        next_in_temp = xgboost_pre_test.iloc[step]["Indoor_Temp"] + next_differ_temp
        xgboost_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp
        data_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp

    state = data_pre_test.iloc[6, :num_features].values

    for step in range(6, 18):
        if fixed_action is None:
            action = epsilon_greedy_policy(q_network, state, epsilon, train_cfg["num_actions"])
        else:
            action = int(fixed_action)

        target_temp, ac_status, window_status, c_last_time, w_last_time = map_action_to_dataframe(action)
        actions.append(action)

        xgboost_pre_test.at[step, "Target_Temp"] = target_temp
        xgboost_pre_test.at[step, "AC_Status"] = ac_status
        xgboost_pre_test.at[step, "Window_Status"] = window_status
        xgboost_pre_test.at[step, "CLast_Time"] = c_last_time
        xgboost_pre_test.at[step, "WLast_Time"] = w_last_time
        xgboost_pre_test.at[step, "CLast_Time_T"] = (
            xgboost_pre_test.iloc[step - 1]["CLast_Time_T"] + c_last_time if c_last_time > 0 else 0
        )
        xgboost_pre_test.at[step, "WLast_Time_T"] = (
            xgboost_pre_test.iloc[step - 1]["WLast_Time_T"] + w_last_time if w_last_time > 0 else 0
        )

        hour_row_df = pd.DataFrame(xgboost_pre_test.iloc[step]).T
        next_differ_temp = model_xgb.predict(hour_row_df)[0]
        next_in_temp = xgboost_pre_test.iloc[step]["Indoor_Temp"] + next_differ_temp
        xgboost_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp
        data_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp

        next_state = data_pre_test.iloc[step + 1, :num_features].values
        reward = calculate_reward(state, action, next_state)
        rewards.append(reward)

        if train and replay_buffer is not None:
            replay_buffer.push(state, action, next_state, reward)
            if len(replay_buffer) >= train_cfg["batch_size"]:
                losses.append(
                    update_q_network(
                        q_network,
                        replay_buffer,
                        train_cfg["optimizer"],
                        train_cfg["loss_fn"],
                        train_cfg["gamma"],
                        train_cfg["num_actions"],
                        train_cfg["batch_size"],
                    )
                )
        state = next_state

    return data_pre_test, actions, rewards, losses

## 10. Hàm tổng hợp metric

Các metric chính dùng để so sánh controller:

- `comfort_pct`: phần trăm timestep nằm trong vùng tiện nghi.
- `ac_on_pct`: phần trăm timestep bật điều hòa.
- `window_open_pct`: phần trăm timestep mở cửa sổ.
- `mean_indoor_temp`: nhiệt độ trong nhà trung bình.
- `total_reward`: tổng reward trong rollout.

Human baseline được lấy trực tiếp từ dữ liệu gốc, còn các controller khác được rollout lại bằng XGBoost.

In [ ]:
def summarize_day(data_day, actions):
    control = data_day.iloc[6:18].copy()
    comfort = [comfort_ok(row["Indoor_Temp"], row["Outdoor_Temp"]) for _, row in control.iterrows()]
    ac_on = sum(1 for a in actions if 0 < a < 12 or a > 12)
    window_open = sum(1 for a in actions if a >= 12)

    return {
        "comfort_pct": float(np.mean(comfort) * 100),
        "ac_on_pct": float(ac_on / len(actions) * 100) if actions else 0.0,
        "window_open_pct": float(window_open / len(actions) * 100) if actions else 0.0,
        "mean_indoor_temp": float(control["Indoor_Temp"].mean()),
    }


def summarize_human(data_test):
    control = data_test.iloc[6:18].copy()
    comfort = [comfort_ok(row["Indoor_Temp"], row["Outdoor_Temp"]) for _, row in control.iterrows()]
    return {
        "comfort_pct": float(np.mean(comfort) * 100),
        "mean_indoor_temp": float(control["Indoor_Temp"].mean()),
    }


def summarize_human_with_actions(data_test, xgboost_test):
    metrics = summarize_human(data_test)
    control = xgboost_test.iloc[6:18].copy()
    metrics["ac_on_pct"] = float(control["AC_Status"].mean() * 100)
    metrics["window_open_pct"] = float(control["Window_Status"].mean() * 100)
    return metrics


def aggregate_metric_rows(rows):
    if not rows:
        return {}
    keys = sorted({key for row in rows for key in row if isinstance(row.get(key), (int, float, np.number))})
    return {key: float(np.mean([row[key] for row in rows if key in row])) for key in keys}

## 11. Huấn luyện DQN trên toàn bộ tập ngày

Ở bản notebook này, phần huấn luyện không cố định vào một lát cắt dữ liệu nhỏ. Thay vào đó, mỗi episode lấy một ngày hoàn chỉnh trong dataset.

Cách chạy mặc định:

- `max_complete_days` là số ngày hoàn chỉnh có thể rollout.
- `EPISODES = max_complete_days`, tức agent đi qua toàn bộ tập ngày một lượt.
- Ở episode thứ `k`, notebook lấy ngày `k % max_complete_days`.
- Với mỗi ngày, agent rollout từ 6h đến 17h, lưu transition vào replay buffer và cập nhật Q-network khi buffer đủ batch.

Thiết kế này giúp notebook thể hiện pipeline toàn bộ rõ hơn:

- XGBoost được train trên toàn bộ dữ liệu sau xử lý.
- DQN được train qua nhiều điều kiện ngày khác nhau.
- Evaluation sau đó cũng chạy trên toàn bộ các ngày hoàn chỉnh.

Nếu muốn rút ngắn thời gian chạy khi demo, có thể đặt `EPISODES = 100` hoặc thấp hơn. Nếu muốn train lâu hơn, có thể đặt `EPISODES = max_complete_days * 2` hoặc lớn hơn để agent đi qua dataset nhiều vòng.


In [ ]:
num_actions = 24
EPISODES = max_complete_days

q_network = DQN(num_actions)
q_network(np.zeros((1, 8), dtype=np.float32))

replay_buffer = ReplayBuffer(10000)
optimizer = tf.optimizers.Adam(0.001)
loss_fn = tf.losses.MeanSquaredError()

epsilon = 1.0
min_epsilon = 0.1
epsilon_decay = 0.995

train_cfg = {
    "num_actions": num_actions,
    "batch_size": 32,
    "gamma": 0.9,
    "optimizer": optimizer,
    "loss_fn": loss_fn,
}

history = []

for episode in range(EPISODES):
    train_day_index = episode % max_complete_days
    train_data_test, train_xgboost_test = choose_day(train_day_index * 24, data)

    _, actions, rewards, losses = rollout_day(
        model_xgb,
        q_network,
        train_data_test,
        train_xgboost_test,
        epsilon,
        train=True,
        replay_buffer=replay_buffer,
        train_cfg=train_cfg,
    )

    if epsilon > min_epsilon:
        epsilon *= epsilon_decay

    history.append(
        {
            "episode": episode + 1,
            "day_index": train_day_index,
            "reward": float(np.sum(rewards)),
            "epsilon": float(epsilon),
            "avg_loss": float(np.mean(losses)) if losses else np.nan,
        }
    )

    if (episode + 1) % 100 == 0 or episode + 1 == EPISODES:
        print(
            f"Episode {episode + 1}/{EPISODES}, "
            f"day={train_day_index}, reward={np.sum(rewards):.2f}, epsilon={epsilon:.3f}"
        )

history_df = pd.DataFrame(history)
history_df.tail()


## 12. Đánh giá toàn bộ pipeline trên tất cả ngày hoàn chỉnh

Sau khi DQN đã được train, notebook đánh giá trên toàn bộ `max_complete_days` ngày hoàn chỉnh.

Các controller được so sánh:

- `DQN`: chính sách học được từ Q-network.
- `Human`: hành vi người dùng có trong dữ liệu gốc.
- `Off_Closed`: luôn tắt điều hòa và đóng cửa.
- `Window_Open`: luôn tắt điều hòa và mở cửa.
- `AC_25_Closed`: luôn bật điều hòa ở 25 độ C và đóng cửa.
- `AC_26_Closed`: luôn bật điều hòa ở 26 độ C và đóng cửa.
- `AC_27_Closed`: luôn bật điều hòa ở 27 độ C và đóng cửa.

Ý nghĩa của bảng đánh giá:

- `comfort_pct` cho biết chính sách giữ nhiệt độ trong vùng tiện nghi tốt đến đâu.
- `ac_on_pct` cho biết mức độ sử dụng điều hòa.
- `window_open_pct` cho biết mức độ dùng thông gió tự nhiên.
- `mean_indoor_temp` giúp kiểm tra xu hướng nhiệt độ trung bình.
- `total_reward` phản ánh trực tiếp hàm reward đang dùng, nên rất nhạy với chi phí bật AC.

Đây là phần quan trọng nhất của notebook vì kết quả được tổng hợp trên toàn bộ các ngày hoàn chỉnh, phù hợp hơn với mục tiêu đánh giá pipeline đầy đủ.


In [ ]:
def evaluate_many_days(model_xgb, q_network, data, day_indices, num_actions):
    controllers = {
        "DQN": None,
        "Human": "human",
        "Off_Closed": 0,
        "Window_Open": 12,
        "AC_25_Closed": 6,
        "AC_26_Closed": 7,
        "AC_27_Closed": 8,
    }
    rows_by_controller = {name: [] for name in controllers}
    actions_by_controller = {name: [] for name in controllers}

    for day_index in day_indices:
        data_test_i, xgboost_test_i = choose_day(day_index * 24, data)

        for name, fixed_action in controllers.items():
            if fixed_action == "human":
                rows_by_controller[name].append(summarize_human_with_actions(data_test_i, xgboost_test_i))
                continue

            eval_day_i, actions_i, rewards_i, _ = rollout_day(
                model_xgb,
                q_network,
                data_test_i,
                xgboost_test_i,
                epsilon=0.0,
                train=False,
                train_cfg={"num_actions": num_actions},
                fixed_action=fixed_action,
            )
            metrics = summarize_day(eval_day_i, actions_i)
            metrics["total_reward"] = float(np.sum(rewards_i))
            rows_by_controller[name].append(metrics)
            actions_by_controller[name].extend(int(action) for action in actions_i)

    summary = []
    action_distributions = {}
    for name, rows in rows_by_controller.items():
        metrics = aggregate_metric_rows(rows)
        summary.append({"controller": name, **metrics})

        actions = actions_by_controller.get(name, [])
        if actions:
            counts = pd.Series(actions).value_counts().sort_index()
            action_distributions[name] = {
                int(action): float(count / len(actions) * 100)
                for action, count in counts.items()
            }

    return summary, action_distributions


eval_indices = list(range(max_complete_days))
evaluation_summary, evaluation_action_distribution = evaluate_many_days(
    model_xgb,
    q_network,
    data,
    eval_indices,
    num_actions,
)

evaluation_df = pd.DataFrame(evaluation_summary)
evaluation_df


## 13. Lưu kết quả full pipeline

Sau khi chạy xong train và evaluate toàn bộ, notebook lưu các artifact sau:

- `training_history.csv`: reward, epsilon, loss trung bình và ngày dùng ở từng episode huấn luyện.
- `summary_metrics.csv`: bảng so sánh các controller trên toàn bộ ngày hoàn chỉnh.
- `result.json`: file tổng hợp toàn bộ metric chính để có thể đọc lại khi viết báo cáo.
- `whulx_dqn_reproduction.weights.h5`: trọng số của Q-network sau huấn luyện.

Với Git, nên commit `result.json`, `summary_metrics.csv`, `training_history.csv` nếu muốn người đọc thấy ngay kết quả full run mà không cần chạy lại notebook. File weight `.h5` có thể bỏ qua nếu muốn repo nhẹ hơn.


In [ ]:
result = {
    "episodes": EPISODES,
    "train_days": max_complete_days,
    "raw_shape": list(raw.shape),
    "after_clean_shape": list(data.shape),
    "xgboost": xgb_metrics,
    "reported_whulx_readme": {
        "comfort_duration_increase_pct": 24.0,
        "ac_usage_decrease_pct": 24.7,
    },
    "evaluation": {
        "eval_days": max_complete_days,
        "controllers": evaluation_summary,
        "action_distribution_pct": evaluation_action_distribution,
    },
}

history_df.to_csv(OUT_DIR / "training_history.csv", index=False)
evaluation_df.to_csv(OUT_DIR / "summary_metrics.csv", index=False)
(OUT_DIR / "result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
q_network.save_weights(OUT_DIR / "whulx_dqn_reproduction.weights.h5")

print("Đã lưu kết quả full pipeline vào:", OUT_DIR)
print(json.dumps(result, indent=2, ensure_ascii=False))


## 14. Cách đọc và diễn giải kết quả

Sau khi chạy notebook, bảng `evaluation_df` là kết quả quan trọng nhất.

Cách diễn giải từng cột:

- `controller`: tên chính sách điều khiển hoặc baseline.
- `comfort_pct`: phần trăm thời điểm nhiệt độ trong nhà nằm trong vùng tiện nghi ASHRAE. Giá trị càng cao càng tốt.
- `ac_on_pct`: phần trăm thời điểm điều hòa được bật. Giá trị thấp hơn thường tương ứng chi phí năng lượng thấp hơn, nhưng có thể làm giảm comfort.
- `window_open_pct`: phần trăm thời điểm cửa sổ được mở. Chỉ số này cho biết chính sách tận dụng thông gió tự nhiên nhiều hay ít.
- `mean_indoor_temp`: nhiệt độ trong nhà trung bình trong khung điều khiển.
- `total_reward`: reward trung bình theo mỗi ngày rollout. Vì reward đang phạt AC khá mạnh, controller bật AC liên tục có thể có comfort cao nhưng total reward rất thấp.

Khi so sánh các controller, không nên chỉ nhìn một metric riêng lẻ. Ví dụ:

- Nếu chỉ tối ưu `comfort_pct`, các baseline bật AC cố định có thể trông tốt hơn.
- Nếu chỉ tối ưu `total_reward`, chính sách tránh dùng AC có thể được ưu tiên quá mức.
- Mục tiêu thực tế là cân bằng giữa comfort và năng lượng, nên cần đọc đồng thời `comfort_pct`, `ac_on_pct` và `total_reward`.

Điểm cần ghi trong báo cáo:

- XGBoost đóng vai trò mô hình dự đoán chuyển trạng thái, không phải controller.
- DQN học controller dựa trên reward.
- Khi train/evaluate trên toàn bộ tập ngày, kết quả phản ánh xu hướng tổng quát hơn so với kiểm tra một ngày riêng lẻ.
- Nếu DQN có xu hướng không bật AC, nguyên nhân chính thường nằm ở thiết kế reward đang phạt năng lượng mạnh.
- Hướng cải tiến tiếp theo là tinh chỉnh reward, thêm ràng buộc comfort hoặc chuyển sang mô phỏng EnergyPlus/Sinergym để có tín hiệu năng lượng vật lý rõ hơn.


## 15. Giải thích kết quả đạt được

Sau khi chạy toàn bộ notebook, kết quả cần tập trung đọc ở `evaluation_df` và file `artifacts/outputs/whulx_reproduction/result.json`.

### Kết quả của XGBoost

XGBoost được dùng để dự đoán `Differ_Indoor_Temp`, tức mức thay đổi nhiệt độ trong nhà ở timestep kế tiếp. Nếu XGBoost có MAE/RMSE thấp, điều đó cho thấy mô hình chuyển trạng thái đủ ổn để dùng làm môi trường gần đúng cho DQN.

Với kết quả đã chạy trước đó trong workspace:

- Dữ liệu gốc có 42,934 dòng và 27 cột.
- Sau làm sạch còn 42,338 dòng và 27 cột.
- XGBoost dùng 21 biến đầu vào.
- MAE khoảng 0.1823 độ C.
- RMSE khoảng 0.3482 độ C.
- R2 khoảng 0.4797.

Cách hiểu: sai số trung bình dưới 0.2 độ C là khá nhỏ cho bài toán dự đoán biến thiên nhiệt độ theo timestep. Tuy nhiên R2 khoảng 0.48 cho thấy XGBoost mới giải thích được một phần biến thiên của dữ liệu, nên đây vẫn là mô hình môi trường gần đúng, chưa phải mô phỏng vật lý hoàn chỉnh.

### Kết quả của DQN

DQN học chính sách chọn action dựa trên reward. Reward hiện tại phạt rất mạnh việc bật điều hòa, đặc biệt khi vừa bật AC vừa mở cửa. Vì vậy, nếu DQN có xu hướng chọn action không bật AC, đây không phải lỗi chạy notebook mà là hệ quả trực tiếp của thiết kế reward.

Với kết quả full evaluation đã chạy trước đó:

- DQN đạt khoảng 67.99% comfort trên 1,763 ngày hoàn chỉnh.
- AC on ratio của DQN là 0.00%.
- Window open ratio của DQN khoảng 34.09%.
- Human baseline đạt comfort cao hơn DQN, khoảng 72.84%, nhưng có dùng AC và mở cửa theo hành vi người thật trong dữ liệu.
- Các baseline bật AC cố định có thể đạt comfort cao hơn, nhưng total reward thường rất thấp vì bị phạt chi phí năng lượng lớn.

Cách hiểu: DQN đã học được chính sách tiết kiệm năng lượng theo reward hiện tại, nhưng chưa tối ưu comfort tốt bằng human baseline hoặc một số baseline bật AC cố định. Đây là kết quả hợp lý cho giai đoạn tái hiện pipeline, vì mục tiêu hiện tại là chứng minh luồng XGBoost + DQN chạy được trên toàn bộ dataset.

### Kết luận nên ghi vào báo cáo

Pipeline đã đạt được ba kết quả chính:

- Tái hiện được phần dự đoán chuyển trạng thái bằng XGBoost trên dữ liệu WHU-LX.
- Xây dựng được DQN tự chọn action HVAC/cửa sổ dựa trên state, reward và transition do XGBoost dự đoán.
- Chạy được đánh giá toàn bộ nhiều ngày, không chỉ kiểm tra một lát cắt dữ liệu nhỏ.

Hạn chế hiện tại:

- Reward đang ưu tiên tiết kiệm AC mạnh, nên agent có xu hướng không bật điều hòa.
- XGBoost chỉ là surrogate model từ dữ liệu, chưa thay thế được mô phỏng vật lý như EnergyPlus.
- Chưa có tín hiệu năng lượng HVAC vật lý chi tiết, nên chi phí năng lượng trong reward vẫn là công thức xấp xỉ.

Hướng cải tiến:

- Điều chỉnh lại reward để cân bằng hơn giữa comfort và năng lượng.
- Thử thêm penalty khi comfort thấp kéo dài nhiều timestep.
- Train DQN nhiều vòng hơn qua toàn bộ dataset.
- Chuyển sang EnergyPlus/Sinergym để đánh giá năng lượng và comfort bằng mô phỏng vật lý rõ hơn.


## 16. So sánh với repo gốc WHU-LX

Phần này dùng để trả lời câu hỏi: kết quả notebook hiện tại có giống kết quả repo gốc hay không, và nếu khác thì khác ở điểm nào.

### Repo gốc công bố điều gì?

Repo WHU-LX/Hvac-Window-based-XGB-DQN đi kèm nghiên cứu về điều khiển HVAC và cửa sổ theo hướng occupant-centric. Theo README/bài báo gốc, hai KPI nổi bật là:

- **Comfort duration tăng khoảng 24%**.
- **AC usage giảm khoảng 24.7%**.

Đây là kết quả tổng hợp ở phạm vi thí nghiệm của nhóm tác giả, không chỉ là kết quả của một lần chạy đơn giản trên một lát cắt dữ liệu.

### Notebook hiện tại đang tái hiện phần nào?

Notebook hiện tại tái hiện lại logic cốt lõi của pipeline:

- Dùng `Cleaned_data.csv` từ WHU-LX.
- Huấn luyện XGBoost để dự đoán `Differ_Indoor_Temp`.
- Dùng XGBoost làm mô hình chuyển trạng thái gần đúng.
- Huấn luyện DQN để chọn action HVAC/cửa sổ.
- Đánh giá DQN trên toàn bộ các ngày hoàn chỉnh trong dataset.

Như vậy, notebook tái hiện được **cấu trúc thuật toán** XGBoost + DQN, nhưng chưa chắc tái hiện hoàn toàn quy trình thí nghiệm và cách tổng hợp KPI giống hệt bài báo gốc.

### Bảng so sánh nhanh

| Nội dung | Repo gốc WHU-LX | Notebook hiện tại |
|---|---|---|
| Dữ liệu | `Cleaned_data.csv` / dữ liệu occupant behavior | `data/Cleaned_data.csv` lấy từ WHU-LX |
| Mô hình chuyển trạng thái | XGBoost dự đoán biến thiên nhiệt độ trong nhà | XGBoost dự đoán `Differ_Indoor_Temp` |
| Controller | DQN chọn action HVAC/cửa sổ | DQN chọn 24 action HVAC/cửa sổ |
| Comfort reference | ASHRAE adaptive comfort | ASHRAE adaptive comfort |
| KPI repo gốc nêu | Comfort duration tăng 24%, AC usage giảm 24.7% | Đánh giá lại bằng `comfort_pct`, `ac_on_pct`, `window_open_pct`, `total_reward` |
| Phạm vi đánh giá | Theo thiết lập thí nghiệm của tác giả | Toàn bộ ngày hoàn chỉnh cắt được từ dataset hiện có |
| Kết luận | Chính sách cải thiện comfort và giảm AC usage theo báo cáo gốc | Pipeline chạy được, nhưng reward hiện tại khiến DQN rất thiên về không bật AC |

### Kết quả có khác repo gốc không?

Có khác ở cách đọc KPI.

Trong kết quả full evaluation đã chạy trước đó, DQN có xu hướng:

- `ac_on_pct` gần 0%, tức gần như không bật điều hòa.
- `window_open_pct` khoảng 34.09%, tức dùng mở cửa ở một phần timestep.
- `comfort_pct` khoảng 67.99%, thấp hơn human baseline khoảng 72.84%.

So với tuyên bố của repo gốc là tăng comfort duration 24% và giảm AC usage 24.7%, notebook hiện tại **chưa tái hiện được đầy đủ kết luận tăng comfort**. Tuy nhiên, notebook có tái hiện rõ xu hướng giảm dùng AC, thậm chí giảm rất mạnh do reward phạt AC lớn.

Điểm cần chú ý: không nên kết luận ngay rằng repo gốc sai hoặc notebook sai. Sự khác biệt có thể đến từ cách thiết lập thí nghiệm.

### Vì sao kết quả có thể khác?

Một số nguyên nhân hợp lý:

- **Khác phạm vi đánh giá**: repo gốc có thể tổng hợp theo cách khác, còn notebook hiện tại đánh giá toàn bộ ngày hoàn chỉnh cắt trực tiếp từ dataset.
- **Khác cách train DQN**: notebook hiện tại train theo pipeline đơn giản, chưa chắc giống hoàn toàn số vòng train, cách chọn ngày, replay strategy hoặc tuning của tác giả.
- **Reward phạt AC rất mạnh**: nếu chi phí bật điều hòa lớn hơn nhiều so với penalty comfort, agent sẽ học cách tránh dùng AC dù comfort chưa tối ưu.
- **Không có target network**: DQN trong notebook là bản tối giản, dùng cùng một Q-network để tính target. DQN chuẩn thường dùng target network để ổn định học.
- **Không chuẩn hóa state**: các biến state có thang đo khác nhau, có thể làm mạng neural học khó hơn.
- **XGBoost là surrogate model**: mô hình dự đoán chuyển trạng thái có sai số, nên rollout nhiều bước có thể tích lũy lỗi.
- **Thiếu mô phỏng vật lý năng lượng**: chi phí năng lượng đang là công thức xấp xỉ, không phải điện năng HVAC tính từ EnergyPlus.

### Kết luận so sánh

Notebook hiện tại tái hiện được pipeline kỹ thuật của repo gốc, nhưng kết quả full evaluation chưa trùng hoàn toàn với KPI công bố của WHU-LX.

Có thể ghi kết luận như sau:

> Kết quả tái hiện cho thấy pipeline XGBoost + DQN có thể chạy được trên dữ liệu WHU-LX và học được chính sách ưu tiên giảm sử dụng điều hòa. Tuy nhiên, trong cấu hình notebook hiện tại, DQN chưa cải thiện comfort so với human baseline trên toàn bộ các ngày đánh giá. Sự khác biệt so với KPI repo gốc có thể đến từ phạm vi đánh giá, cấu hình train DQN, reward phạt năng lượng mạnh và việc notebook dùng bản DQN tối giản. Do đó, kết quả hiện tại nên được xem là baseline tái hiện kỹ thuật, chưa phải bản sao đầy đủ của toàn bộ thí nghiệm trong bài báo.

### Hướng làm để tiến gần repo gốc hơn

Để kiểm tra chặt hơn với repo gốc, có thể làm các bước sau:

- Đọc kỹ notebook gốc `XGB-DQN.ipynb` và đối chiếu từng hyperparameter.
- Train DQN nhiều vòng hơn qua toàn bộ dataset.
- Thêm target network cho DQN.
- Chuẩn hóa state trước khi đưa vào neural network.
- Điều chỉnh reward để giảm mức phạt AC hoặc tăng penalty khi comfort thấp.
- So sánh theo đúng công thức `comfort duration increase` và `AC usage decrease` mà repo gốc dùng.
- Lưu lại seed, action distribution và bảng metric cho từng controller để kiểm chứng từng lần chạy.
